# COMPACT-MI-RFE CICFLOW — luồng ColabPro, kết quả theo experiment

12 cell: runtime → Drive → dataset → source → dependencies/tests → metadata → scientific contract → experiment → preflight → MI/RFE/k-search/final/Test → verify → summary.

Đọc trực tiếp `/content/drive/MyDrive/CICFLOWMETER_FEATURE_CSV`. Mỗi lần chọn **một model + một FW/FS**. Kết quả hoàn tất tại `/content/drive/MyDrive/experiment_results/<run_name>/`, checkpoint tại `/content/drive/MyDrive/checkpoints/<run_name>.incomplete/`.

**PROVISIONAL_RESEARCH**, chưa phải official reproduction. Pipeline này dùng StandardScaler và **một lần gọi MI trên 380 feature seed42** theo source ColabPro; khác Stage 4 cũ dùng seed theo feature. Run cũ được bảo tồn và không được dùng thay kết quả mới. XGBoost BLOCKED vì chưa chốt ranking contract.

Chỉ các cell bạn chủ động chạy mới thực thi. Mặc định `dry_run`. Full run gồm k=1..100, refit Train+Validation rồi Test một lần; có thể lâu, nhất là kernel SVM. Không giảm scientific parameters.


## Cell 1 — Thông tin runtime

Xem Python, thư viện, CPU, RAM và disk như Cell 1 của mẫu. Không huấn luyện; GPU không tăng tốc các estimator sklearn này.


In [ ]:
# CELL 1 — Runtime information
import importlib.metadata as metadata
import json, os, platform, shutil, sys
from pathlib import Path
print("Python:", sys.version)
print("OS:", platform.platform(), "| logical CPUs:", os.cpu_count())
print("Disk /content:", shutil.disk_usage("/content"))
for package in ("numpy", "pandas", "scipy", "scikit-learn", "pyarrow", "psutil", "joblib"):
    try:
        print(package, metadata.version(package))
    except metadata.PackageNotFoundError:
        print(package, "MISSING — install explicitly before Cell 5")
try:
    import psutil
    print("RAM:", psutil.virtual_memory())
except ImportError:
    pass
print("CPU pipeline; no test, fit, dataset copy or automatic pip install in this cell.")


## Cell 2 — Mount Google Drive

Mount Drive theo mẫu; không ép remount nếu Drive đang có.


In [ ]:
# CELL 2 — Mount Google Drive
from google.colab import drive
drive.mount("/content/drive")
if not Path("/content/drive/MyDrive").is_dir():
    raise FileNotFoundError("Google Drive MyDrive is not mounted")


## Cell 3 — Dataset và vị trí lưu

Dùng đúng MyDrive/code_CICFLOW và MyDrive/CICFLOWMETER_FEATURE_CSV. Output dùng MyDrive/experiment_results như ảnh, không dùng COMPACT_MI_RFE_FRESH_RUNS.


In [ ]:
# CELL 3 — Drive paths, no transfer or extraction
DRIVE_ROOT = Path("/content/drive/MyDrive")
SOURCE_ROOT = DRIVE_ROOT / "code_CICFLOW"
DATASET_ROOT = DRIVE_ROOT / "CICFLOWMETER_FEATURE_CSV"
OUTPUT_ROOT = DRIVE_ROOT
CONFIGS = ("FW256_FS8", "FW256_FS16", "FW512_FS8", "FW512_FS16")
for label, folder in (("SOURCE_ROOT", SOURCE_ROOT), ("DATASET_ROOT", DATASET_ROOT)):
    print(label, folder, "FOUND" if folder.is_dir() else "MISSING")
    if not folder.is_dir():
        raise FileNotFoundError(folder)
for config in CONFIGS:
    folder = DATASET_ROOT / config
    print(config, "FOUND" if folder.is_dir() else "MISSING")
    if not folder.is_dir():
        raise FileNotFoundError(folder)
print("Completed results:", OUTPUT_ROOT / "experiment_results")
print("Incomplete checkpoints:", OUTPUT_ROOT / "checkpoints")
print("Old COMPACT_MI_RFE_FRESH_RUNS results will not be changed or reused.")


## Cell 4 — Load đúng source trên Drive

Thay bước Git clone của mẫu bằng source đã upload trên Drive. Nạp chính xác helper mới; không patch source hoặc tải dataset.


In [ ]:
# CELL 4 — Exact source import, no clone/pull/reset or legacy embedded patch
import importlib.util
required = (
    "colab/cicflow_reference_pipeline.py",
    "colab/test_cicflow_reference_pipeline.py",
    "stage4_mi/mi_selection.py",
    "stage5_rfe/rfe_selection.py",
)
for relative in required:
    path = SOURCE_ROOT / relative
    print("PASS" if path.is_file() else "MISSING", path)
    if not path.is_file():
        raise FileNotFoundError(f"Upload required source: {path}")
module_path = SOURCE_ROOT / "colab/cicflow_reference_pipeline.py"
module_spec = importlib.util.spec_from_file_location("cicflow_reference_runtime", module_path)
CICFLOW = importlib.util.module_from_spec(module_spec)
sys.modules[module_spec.name] = CICFLOW
module_spec.loader.exec_module(CICFLOW)
print("Loaded:", CICFLOW.__file__)
print("Format:", CICFLOW.FORMAT)
print("Source SHA256:", CICFLOW.source_snapshot(SOURCE_ROOT)["sha256"])
SETUP = {"source": str(SOURCE_ROOT), "dataset": str(DATASET_ROOT), "output": str(OUTPUT_ROOT)}


## Cell 5 — Dependencies và unit/regression tests

Kiểm tra imports và chạy fixture tests khi bạn chủ động chạy cell. Có regression so score/ranking 380 feature với một lần MI reference, RFE ties/resume và toàn luồng publication bằng dữ liệu giả. Không có bằng chứng PASS nếu chưa thực thi.


In [ ]:
# CELL 5 — Dependency/import evidence and synthetic regression tests
DEPENDENCIES = CICFLOW.launch("dependencies", SETUP)
print(json.dumps(DEPENDENCIES["result"], indent=2))
if DEPENDENCIES["exit_code"] != 0 or DEPENDENCIES["program_status"] != "PASS":
    raise RuntimeError("Missing/broken dependencies. Review log; no parameters were changed.")
TEST_EXECUTION = CICFLOW.launch("tests", SETUP)
UNIT_TEST_EVIDENCE = TEST_EXECUTION["result"]
print(json.dumps(UNIT_TEST_EVIDENCE, indent=2))
if TEST_EXECUTION["exit_code"] != 0 or TEST_EXECUTION["program_status"] != "PASS":
    raise RuntimeError("Tests failed/no tests/skipped: training is BLOCKED. Preserve the log.")
print("Fixture tests PASS is not full-dataset CTU-13 equivalence evidence.")


## Cell 6 — Metadata dataset

Đọc footer Parquet của bốn cấu hình: 380 feature và Window_Label, thứ tự, kiểu, số hàng, file stat và sidecar hashes. Không đọc giá trị Train/Validation/Test, không kết luận leakage PASS.


In [ ]:
# CELL 6 — Dataset metadata only, explicit status for each configuration
DATA_AUDIT = {}
for config in CONFIGS:
    execution = CICFLOW.launch("metadata", {**SETUP, "config": config})
    DATA_AUDIT[config] = execution
    item = execution["result"]
    print("\n", config, execution["program_status"])
    if item.get("binding"):
        print("Features:", len(item["binding"]["names"]))
        print("Rows:", {role: record["rows"] for role, record in item["binding"]["splits"].items()})
        print("Fingerprint:", item["fingerprint"])
    else:
        print(item)
DATASET_READY = all(r["program_status"] == "PASS" and r["exit_code"] == 0 for r in DATA_AUDIT.values())
print("\nDATASET_READY:", DATASET_READY)
if not DATASET_READY:
    raise RuntimeError("Dataset metadata gate failed; see per-configuration logs.")
print("No row values read. Capture/session independence and label validity remain unverified.")


## Cell 7 — Scientific contract

Theo mẫu: Train scaler + một MI call seed42 → Top100 → Train-only scaler/RFE riêng từng nhánh → Validation k-search → freeze → final scaler/model Train+Validation → Test một lần. RF classifier depth20 khác RF-RFE depth10.


In [ ]:
# CELL 7 — Show scientific contract before any fit
print("SCIENTIFIC STATUS:", CICFLOW.SCIENTIFIC_STATUS)
for branch in ("rf", "svm"):
    print("\nBRANCH:", branch)
    print(json.dumps(CICFLOW.contract(branch), indent=2))
print("\nIMPORTANT: previous per-feature Stage 4 scores are not reference-equivalent inputs here.")
print("RF classifier: 300 trees, max_depth=20, min_samples_split=3.")
print("RF-RFE estimator: 100 trees, depth10; SVM-branch RF-RFE estimator: 100 trees, depth15.")
print("SVM classifier: C=10, max_iter=20000; full remaining parameters are recorded above.")
print("XGBoost BLOCKED until its ranking estimator is reconciled; no RF ranking substitution.")


## Cell 8 — Chọn một experiment/configuration

Giống Cell 8 mẫu: chọn model, FW, FS và Run Name. Để Run Name trống chỉ khi tạo run mới; helper chọn runNN chưa tồn tại. Resume/verify phải nhập đúng tên cũ. Mặc định dry_run.


In [ ]:
# CELL 8 — One experiment/config, like the reference notebook
EXPERIMENT = "mi_rfe_rf"  #@param ["mi_rfe_rf", "mi_rfe_svm"]
FW = "256"  #@param ["256", "512"]
FS = 8  #@param [8, 16]
RUN_NAME = ""  #@param {type:"string"}
MODE = "dry_run"  #@param ["dry_run", "full_train", "resume", "verify"]
CONFIRMATION = ""  #@param {type:"string"}
GIT_COMMIT = "NOT_CAPTURED"  #@param {type:"string"}
CONFIG = f"FW{FW}_FS{int(FS)}"
if MODE in ("resume", "verify") and not RUN_NAME:
    raise ValueError("Resume/verify requires the exact existing RUN_NAME; do not generate a new ID.")
if not RUN_NAME:
    RUN_NAME = CICFLOW.next_run_name(OUTPUT_ROOT, EXPERIMENT, CONFIG)
SPEC = CICFLOW.RunSpec(EXPERIMENT, CONFIG, RUN_NAME).validate()
REQUEST = {
    **SETUP, "spec": {"experiment": SPEC.experiment, "config": SPEC.config, "run_name": SPEC.run_name},
    "mode": MODE, "confirmation": CONFIRMATION, "git_commit": GIT_COMMIT,
    "unit_tests": UNIT_TEST_EVIDENCE,
}
PREFLIGHT = None
RUN_RESULT = None
print(json.dumps({k: v for k, v in REQUEST.items() if k != "unit_tests"}, indent=2))
print("Exact confirmation:", "RUN PROVISIONAL " + SPEC.run_name)
print("Result:", OUTPUT_ROOT / "experiment_results" / SPEC.run_name)
print("Checkpoint:", OUTPUT_ROOT / "checkpoints" / (SPEC.run_name + ".incomplete"))


## Cell 9 — Preflight và resource gate

Giống Cell 9 mẫu: fresh worker kiểm tra source, dataset, versions, tests, identity, RAM/disk và non-overwrite. Full/resume cần xác nhận đúng run. Gate PASS là trạng thái kỹ thuật, không chứng minh official reproduction.


In [ ]:
# CELL 9 — Fresh preflight; no fit
if not DATASET_READY:
    raise RuntimeError("Run Cell 6 successfully before preflight")
if MODE in ("full_train", "resume") and not CONFIRMATION:
    phrase = "RUN PROVISIONAL " + SPEC.run_name
    CONFIRMATION = input(f"Gõ chính xác để chạy {SPEC.experiment}/{SPEC.config}:\n{phrase}\n").strip()
REQUEST.update(mode=MODE, confirmation=CONFIRMATION, unit_tests=UNIT_TEST_EVIDENCE, git_commit=GIT_COMMIT)
PREFLIGHT_EXECUTION = CICFLOW.launch("preflight", REQUEST)
PREFLIGHT = PREFLIGHT_EXECUTION["result"]
print(json.dumps(PREFLIGHT, indent=2))
if PREFLIGHT_EXECUTION["exit_code"] != 0 or PREFLIGHT_EXECUTION["program_status"] != "PASS":
    raise RuntimeError("PREFLIGHT BLOCKED. Do not reduce rows, trees, k grid, RFE step or scientific parameters.")
print("Ready for Cell 10. Current mode:", MODE)


## Cell 10 — MI → RFE → k-search → final refit → Test

Giống Cell 10 mẫu: một worker độc lập cho toàn bộ experiment, output live và log lên Drive. K-search đủ 100 giá trị k; Test chỉ mở sau freeze và final refit. Không chạy tất cả cấu hình ngầm. Resume dùng cùng run/identity và checkpoint.


In [ ]:
# CELL 10 — Full experiment in a fresh logged worker
if MODE in ("full_train", "resume"):
    if not PREFLIGHT or PREFLIGHT.get("status") != "PASS":
        raise RuntimeError("Successful current preflight required")
    if REQUEST["spec"] != {"experiment": SPEC.experiment, "config": SPEC.config, "run_name": SPEC.run_name}:
        raise RuntimeError("Experiment changed; rerun Cells 8-9")
    if MODE != REQUEST["mode"]:
        raise RuntimeError("Mode changed; rerun Cell 9")
    print("Starting:", SPEC)
    print("This runs MI, RF-RFE, 100 classifier fits for k-search, final refit, and one Test evaluation.")
    print("Kernel SVM may take substantially longer than RF; no automatic sampling or hyperparameter changes.")
    RUN_RESULT = CICFLOW.launch("run", REQUEST)
    print(json.dumps(RUN_RESULT, indent=2))
    if RUN_RESULT["exit_code"] != 0 or RUN_RESULT["program_status"] != "PASS":
        raise RuntimeError("Experiment failed/BLOCKED. Preserve checkpoint and log; review before resume.")
    print("Published:", RUN_RESULT["result"]["directory"])
elif MODE == "dry_run":
    print("DRY RUN ONLY — no MI, RFE, k-search, final refit or Test evaluation.")
elif MODE == "verify":
    print("VERIFY MODE — no fit. Continue to Cell 11.")
else:
    raise ValueError(MODE)


## Cell 11 — Verify artifact publication

Giống Cell 11 mẫu: kiểm tra hashes/size, 380 MI scores, Top100, đủ 100 rank/k, frozen features, final binding và đúng một Test event theo thứ tự. Không load model và không dự đoán Test lần nữa.


In [ ]:
# CELL 11 — Verify published artifacts without refit or Test read
VERIFY_RESULT = None
if MODE in ("full_train", "resume", "verify"):
    VERIFY_RESULT = CICFLOW.launch("verify", REQUEST)
    print(json.dumps(VERIFY_RESULT, indent=2))
    if VERIFY_RESULT["exit_code"] != 0 or VERIFY_RESULT["program_status"] != "PASS":
        raise RuntimeError("Artifact verification failed. Do not treat this run as completed.")
    print("Drive artifacts verified:", VERIFY_RESULT["result"]["directory"])
else:
    print("Dry run has no completed result to publish or verify.")


## Cell 12 — Tổng hợp result.json và vị trí file

Giống Cell 12 mẫu: hiển thị kết quả đã lưu, không đọc dataset/model. Thư mục hoàn tất có đúng các artifact trong ảnh cùng manifest/identity bổ sung. Lặp Cell 8–12 cho model/config khác bằng Run Name mới.


In [ ]:
# CELL 12 — Saved result summary and exact Drive locations
SUMMARY_EXECUTION = CICFLOW.launch("summary", SETUP)
ROWS = SUMMARY_EXECUTION["result"].get("rows", [])
try:
    import pandas as pd
    frame = pd.DataFrame(ROWS)
    if len(frame):
        columns = [c for c in ("run_name", "experiment", "config", "status", "verify", "best_k",
                               "validation_accuracy", "accuracy", "precision_attack", "recall_attack",
                               "f1_attack", "macro_f1", "balanced_accuracy", "directory") if c in frame]
        display(frame[columns])
    else:
        print("No completed results from this pipeline yet.")
except ImportError:
    print(json.dumps(ROWS, indent=2))
print("\nCompleted experiments:", OUTPUT_ROOT / "experiment_results")
print("Current result:", OUTPUT_ROOT / "experiment_results" / SPEC.run_name)
print("Current checkpoint:", OUTPUT_ROOT / "checkpoints" / (SPEC.run_name + ".incomplete"))
print("Aggregate log:", SOURCE_ROOT / "logs/ALL_EXECUTION_RESULTS.log")
print("Task logs:", SOURCE_ROOT / "logs/colab_reference" / SPEC.run_name)
print("Expected files:", "\n".join(CICFLOW.REQUIRED_ARTIFACTS))
print("\nResume: Cell 8 MODE='resume', exact existing RUN_NAME and original GIT_COMMIT; rerun Cells 9-12.")
print("After a new session run Cells 1-9 first. Completed published runs are verified without any Test reevaluation.")
print("PROVISIONAL_RESEARCH; label/capture/session validity and full-data reference equivalence remain unproven.")
